# Part 3: Predicting No-Shows

Building a model that scores bookings on the evening before the visit, using only training data for model selection and threshold tuning.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score
)

## Task 3.1 — Features, split, and baseline

Keeping `appointments_clean.csv` in its saved row order so the 80/20 split is reproducible.

In [ ]:
df = pd.read_csv("appointments_clean.csv")
appt_ids = df["appointment_id"].copy()

feature_cols = [
    "age", "distance_km", "fee_inr", "lead_days",
    "prior_no_shows", "reminder_sent", "booking_channel", "visit_type"
]
X = df[feature_cols].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_acc = (y_test == 0).mean()
print(f"X shape: {X.shape}")
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")
print(f"Test no-shows: {int(y_test.sum())} / {len(y_test)}")
print(f"Baseline accuracy (always predict Attended): {baseline_acc:.4f}")

## Task 3.2 — Logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

lr_acc = accuracy_score(y_test, lr_pred)
lr_prec = precision_score(y_test, lr_pred)
lr_rec = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_prob)

print(f"Accuracy: {lr_acc:.4f} | Precision: {lr_prec:.4f} | Recall: {lr_rec:.4f} | F1: {lr_f1:.4f} | ROC-AUC: {lr_auc:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": lr.coef_[0],
    "scale": scaler.scale_
})
print("\nCoefficients:\n", coef_df.to_string(index=False))

rem_row = coef_df[coef_df["feature"] == "reminder_sent"].iloc[0]
odds_scaled = np.exp(rem_row["coef"])
odds_unscaled = np.exp(rem_row["coef"] / rem_row["scale"])
print(f"\nreminder_sent exp(coef) [per 1 SD]: {odds_scaled:.3f}")
print(f"reminder_sent exp(coef / scale) [No -> Yes]: {odds_unscaled:.3f} (~{(1 - odds_unscaled)*100:.0f}% lower odds)")

The two odds ratios differ because `StandardScaler` puts `reminder_sent` in standard-deviation units (SD around 0.49), so dividing the coefficient by `scaler.scale_` converts it back to a full 0-to-1 jump. In this model, holding the other features constant, appointments with a reminder sent are associated with roughly 69% lower odds of a no-show (an observational association, not proof of causation).

## Task 3.3 — Decision tree and overfitting

In [ ]:
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
print(f"Full tree (depth={dt_full.get_depth()}): train acc={dt_full.score(X_train, y_train):.4f}, test acc={dt_full.score(X_test, y_test):.4f}")

dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)
dt3_pred = dt_3.predict(X_test)
dt3_prob = dt_3.predict_proba(X_test)[:, 1]
print(f"Depth-3 tree: train acc={dt_3.score(X_train, y_train):.4f}, test acc={dt_3.score(X_test, y_test):.4f}")

The gap between 1.00 training accuracy and ~0.67 test accuracy on the unconstrained tree shows severe overfitting: it memorised training noise and performed worse than always predicting Attended.

## Task 3.4 — Random forest and model comparison

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print(f"RF train acc: {rf.score(X_train, y_train):.4f}")
print(f"RF test acc: {accuracy_score(y_test, rf_pred):.4f} | recall: {recall_score(y_test, rf_pred):.4f} | ROC-AUC: {roc_auc_score(y_test, rf_prob):.4f}")

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 feature importances:\n", imp.head(3))

comp_df = pd.DataFrame([
    {"Model": "Logistic Regression", "Accuracy": lr_acc, "Recall": lr_rec, "ROC-AUC": lr_auc},
    {"Model": "Decision Tree (max_depth=3)", "Accuracy": accuracy_score(y_test, dt3_pred), "Recall": recall_score(y_test, dt3_pred), "ROC-AUC": roc_auc_score(y_test, dt3_prob)},
    {"Model": "Random Forest (100)", "Accuracy": accuracy_score(y_test, rf_pred), "Recall": recall_score(y_test, rf_pred), "ROC-AUC": roc_auc_score(y_test, rf_prob)},
])
comp_df.round(4)

## Task 3.5 — Cross-validated tuning, threshold selection, and single test evaluation

Selecting both the model and the probability threshold strictly from 5-fold CV on the training set before running one final check on `X_test`.

In [ ]:
# (a) Tune Random Forest on training set
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc"
)
grid.fit(X_train, y_train)
print("GridSearchCV best_params_:", grid.best_params_)
print(f"GridSearchCV best_score_ (CV ROC-AUC): {grid.best_score_:.4f}")

# (b) Compare 5-fold CV ROC-AUC on training set
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_tuned_rf = grid.best_score_

print(f"CV ROC-AUC -> LR: {cv_lr:.4f} | DT(3): {cv_dt3:.4f} | RF: {cv_rf:.4f} | Tuned RF: {cv_tuned_rf:.4f}")
print("Tuned RF test ROC-AUC (info only):", round(roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1]), 4))

In [ ]:
# (c) Out-of-fold probabilities for the winning model (Logistic Regression pipeline)
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

for thresh in [0.5, 0.35]:
    oof_flag = (oof >= thresh).astype(int)
    print(
        f"Threshold {thresh}: "
        f"precision={precision_score(y_train, oof_flag):.4f}, "
        f"recall={recall_score(y_train, oof_flag):.4f}, "
        f"F1={f1_score(y_train, oof_flag):.4f}, "
        f"flagged={int(oof_flag.sum())}"
    )

chosen_threshold = 0.35

# (d) Fit winner on full training set and evaluate once on test set
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flags = (probs >= chosen_threshold).astype(int)

print("\n--- Final Test Evaluation (threshold = 0.35) ---")
print(f"Precision: {precision_score(y_test, test_flags):.4f}")
print(f"Recall:    {recall_score(y_test, test_flags):.4f}")
print(f"Flagged:   {int(test_flags.sum())} / {len(y_test)}")
print(f"ROC-AUC:   {roc_auc_score(y_test, probs):.4f}")

# (e) Save test_risk_scores.csv
scores_df = pd.DataFrame({
    "appointment_id": appt_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": np.round(probs, 4),
    "flag": test_flags,
})
scores_df.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv with rows:", len(scores_df))

Recall matters more than raw accuracy here because missing a likely no-show wastes an entire doctor slot, whereas a false alarm only costs the front desk a one-minute confirmation call. In practice, a clinic would set this threshold based on how many calls the evening desk can realistically make compared with the cost of an empty slot.